In [0]:
import requests
import time
from datetime import datetime, timezone
from typing import Optional
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)

In [0]:
BASE_URL   = "https://v6.exchangerate-api.com/v6/{api_key}/latest/{base}"
BASE_CCY   = "GBP"
PAIR_CCYS  = ["USD","EUR","AED","NGN","CNY","CHF","JPY","SGD"]
MAX_RETRY  = 3
RETRY_WAIT = 5  # seconds

def fetch_fx_rates(api_key: str,
                   base: str = BASE_CCY) -> Optional[dict]:
    url = BASE_URL.format(api_key=api_key, base=base)
    for attempt in range(1, MAX_RETRY + 1):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            if data.get("result") != "success":
                raise ValueError(
                    f"API error: {data.get('error-type','unknown')}")
            return data
        except Exception as e:
            if attempt == MAX_RETRY:
                raise RuntimeError(
                    f"FX API failed after {MAX_RETRY} attempts: {e}")
            time.sleep(RETRY_WAIT * attempt)

def parse_rates(data: dict,
                pair_ccys: list = PAIR_CCYS) -> list[dict]:
    rates     = data["conversion_rates"]
    base      = data["base_code"]
    source_ts = data["time_last_update_utc"]
    fetch_ts  = datetime.now(timezone.utc)

    return [
        {
            "rate_id":        f"{base}{ccy}_{fetch_ts.strftime('%Y%m%d%H%M')}",
            "base_currency":  base,
            "quote_currency": ccy,
            "rate":           float(rates[ccy]),
            "source_ts":      source_ts,
            "fetch_ts":       fetch_ts,
            "provider":       "exchangerate-api",
            "pipeline_version": "1.0.0",
        }
        for ccy in pair_ccys
        if ccy in rates
    ]

In [0]:


api_key = dbutils.secrets.get(
    scope="market-data", key="fx-api-key")
print(f"FX API key: {api_key}")

VALID_QUOTE_CCYS = ["USD","EUR","AED","NGN","CNY","CHF","JPY","SGD"]

raw_data = fetch_fx_rates(api_key, base="GBP")
records  = parse_rates(raw_data, VALID_QUOTE_CCYS)

In [0]:

BASE_CCY     = "GBP"
PAIR_CCYS    = ["USD","EUR","AED","NGN","CNY","CHF","JPY","SGD"]
STAGING_TABLE = "banking_demo.bronze.fx_rates_stage"
MAX_RETRY    = 3

FX_RATE_SCHEMA = StructType([
    StructField("rate_id",          StringType(),    False),
    StructField("base_currency",    StringType(),    False),
    StructField("quote_currency",   StringType(),    False),
    StructField("rate",             DoubleType(),    False),
    StructField("source_ts",        StringType(),    True),
    StructField("fetch_ts",         TimestampType(), False),
    StructField("provider",         StringType(),    False),
    StructField("pipeline_version", StringType(),    False),
])

In [0]:
df = spark.createDataFrame(records, schema=FX_RATE_SCHEMA)

df.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable(STAGING_TABLE)

row_count = df.count()
print(f"Wrote {row_count} rows to {STAGING_TABLE}")
dbutils.jobs.taskValues.set("staged_row_count", row_count)
